In [ ]:
import pandas as pd
# from src.weather import fetch_weather_month, RENAME

# df = pd.DataFrame(fetch_weather_month["hourly"]).rename(columns=RENAME)
# df.head()



TypeError: 'function' object is not subscriptable

In [7]:
import requests

YEAR = 2024
MONTH = 1

API_URL = "https://archive-api.open-meteo.com/v1/archive"
NYC_LAT, NYC_LON = 40.7128, -74.006
HOURLY_VARS = "temperature_2m,precipitation,rain,snowfall,wind_speed_10m"
TIMEZONE = "America/New_York"

RENAME = {
    "time": "observed_hour",
    "temperature_2m": "temperature_c",
    "precipitation": "precipitation_mm",
    "rain": "rain_mm",
    "snowfall": "snowfall_cm",
    "wind_speed_10m": "wind_speed_kmh",
}
VALUE_COLS = ["temperature_c", "precipitation_mm", "rain_mm",
              "snowfall_cm", "wind_speed_kmh"]

start = pd.Timestamp(year=YEAR, month=MONTH, day=1)
end = start + pd.offsets.MonthEnd(0)
if end >= pd.Timestamp.now():
    raise ValueError(
        f"Month {YEAR}-{MONTH:02d} has not fully ended; "
        "refusing to ingest a partial month."
    )
response = requests.get(
    API_URL,
    params={
        "latitude": NYC_LAT, "longitude": NYC_LON,
        "start_date": start.date().isoformat(),
        "end_date": end.date().isoformat(),
        "hourly": HOURLY_VARS, "timezone": TIMEZONE,
    },
    timeout=60,
)
response.raise_for_status()
payload = response.json()
if payload.get("error"):
    raise ValueError(f"Open-Meteo error: {payload.get('reason')}")
if "hourly" not in payload:
    raise ValueError("Open-Meteo response missing 'hourly' section")


In [8]:
df = pd.DataFrame(payload["hourly"]).rename(columns=RENAME)

In [9]:
df.head()

,observed_hour,temperature_c,precipitation_mm,rain_mm,snowfall_cm,wind_speed_kmh
0,2024-01-01T00:00,1.9,0.0,0.0,0.0,6.3
1,2024-01-01T01:00,1.8,0.0,0.0,0.0,8.9
2,2024-01-01T02:00,2.8,0.0,0.0,0.0,11.4
3,2024-01-01T03:00,2.9,0.0,0.0,0.0,9.7
4,2024-01-01T04:00,2.8,0.0,0.0,0.0,8.1


In [10]:
print("DataFrame loaded")
print("Rows: %s", len(df))
print("Columns: %s", len(df.columns))
print("Column names: %s", list(df.columns))

missing = df.isna().sum()
if missing.any():
    print("Missing values:\n%s", missing[missing > 0].sort_values(ascending=False))
else:
    print("No missing values found.")

DataFrame loaded
Rows: %s 744
Columns: %s 6
Column names: %s ['observed_hour', 'temperature_c', 'precipitation_mm', 'rain_mm', 'snowfall_cm', 'wind_speed_kmh']
No missing values found.
